# 02: Generation Pipeline

Phase 1. LangGraph pipeline for AML query answering using dense+sparse retrieval. Builds the StateGraph, citation validator, and guardrails. Runs a smoke test without loading the LLM.

Loading the LLM requires `HF_TOKEN` and a GPU; every other cell runs without it.

## Cell group 1: Imports and constants

In [1]:
from dotenv import load_dotenv
load_dotenv()

import json
import re
import sys
from pathlib import Path
from typing import Optional, TypedDict
import torch

SCRIPTS_DIR = Path('scripts').resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from retriever import load_dense_sparse, dense_sparse_retrieve
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline_fn
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langgraph.graph import StateGraph, END

DATA_DIR    = Path('data')
CHROMA_DIR  = DATA_DIR / 'chroma_db'

MODEL_ID         = 'NousResearch/Meta-Llama-3.1-8B-Instruct'  # fixed regardless of hardware -
                                                              # generator identity must stay
                                                              # constant across local/cluster
                                                              # runs (see plan.md Sec 5,
                                                              # "Generator model identity")
LOAD_IN_4BIT     = True   # True = NF4 4-bit for 6GB local GPU; False = BF16 for H200 cluster (same model, just more VRAM headroom)
MAX_NEW_TOKENS   = 1024
TOP_K_RETRIEVAL  = 10
GRAPH_HOPS       = 2
RRF_K            = 60
MAX_RETRIES      = 2
GUARDRAILS_ON    = False

print(f'MODEL_ID       : {MODEL_ID}')
print(f'LOAD_IN_4BIT   : {LOAD_IN_4BIT}  (set False on H200 cluster for BF16, same model)')
print(f'TOP_K_RETRIEVAL: {TOP_K_RETRIEVAL}')
print(f'GUARDRAILS_ON  : {GUARDRAILS_ON}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name  = torch.cuda.get_device_name(0)
    vram  = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f'GPU            : {name}  ({vram}GB VRAM)')

MODEL_ID       : NousResearch/Meta-Llama-3.1-8B-Instruct
LOAD_IN_4BIT   : True  (set False on H200 cluster for BF16 + 70B)
TOP_K_RETRIEVAL: 10
GUARDRAILS_ON  : False
CUDA available : True
GPU            : NVIDIA GeForce RTX 3050 6GB Laptop GPU  (5GB VRAM)


In [ ]:
from log_setup import setup_logging

logger = setup_logging("02_generation_pipeline")

## Cell group 2: Load hybrid retriever

Loads all three indices. ChromaDB reloads from disk. BM25 and NetworkX rebuild from JSONL files (~0.5s total).

In [3]:
import time
from retriever import load_dense_sparse, dense_sparse_retrieve

t0 = time.time()
vectorstore, bm25, clauses = load_dense_sparse(DATA_DIR, CHROMA_DIR)
clause_lookup = {c['clause_id']: c for c in clauses}
logger.info('Retrievers loaded: %d clauses  (%.1fs)', len(clauses), time.time() - t0)
print(f'Loaded {len(clauses)} clauses  ({time.time()-t0:.1f}s)')

03:44:17  INFO      Loading dense+sparse retrievers from data
03:44:19  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
03:44:19  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
03:44:19  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
03:44:19  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
03:44:19  INFO      Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
03:44:19  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transfor

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

03:44:20  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
03:44:20  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
03:44:20  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
03:44:20  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
03:44:20  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:44:21  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/toke

## Cell group 3: Prompt and LCEL chain

Grounding contract: every claim must cite a clause as `[clause_id]`. Output must be valid JSON with `answer` and `citations` fields.

In [4]:
SYSTEM_PROMPT = '''You are an AML compliance analyst. Answer using ONLY the regulatory clauses below.

Rules:
1. Every factual claim must be followed by [clause_id] inline.
2. State explicitly if the provided context does not answer the question.
3. Output valid JSON with two keys:
   - answer: your response with inline [clause_id] citations
   - citations: list of clause_id strings you cited

Context:
{context}'''


def format_context(results: list) -> str:
    parts = []
    for r in results:
        cid  = r['clause_id']
        text = r['text'][:800]
        parts.append(f'[{cid}]\n{text}')
    return '\n\n---\n\n'.join(parts)


prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{query}'),
])

print('Prompt template defined.')
print()
print('To load the model, run the cell below.')
print('Prerequisites:')
print('  1. Accept the Meta licence at huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct')
print('  2. Set HF_TOKEN env var:  setx HF_TOKEN hf_xxxxx  (then restart kernel)')
print(f'  3. CUDA must be available: {torch.cuda.is_available()}')

Prompt template defined.

To load the model, run the cell below.
Prerequisites:
  1. Accept the Meta licence at huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct
  2. Set HF_TOKEN env var:  setx HF_TOKEN hf_xxxxx  (then restart kernel)
  3. CUDA must be available: True


In [5]:
import os, torch

_hf_token = os.getenv("HF_TOKEN", "")
_has_cuda  = torch.cuda.is_available()

if not _hf_token:
    raise EnvironmentError("HF_TOKEN not set -- add it to .env and restart the kernel.")
if not _has_cuda:
    raise EnvironmentError("No CUDA device found -- this notebook requires a GPU.")

# Free any cached VRAM before loading the model
torch.cuda.empty_cache()

if LOAD_IN_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    # Force all layers onto cuda:0, avoids split-device rejection from bitsandbytes
    model_kwargs = {"quantization_config": quant_config, "device_map": {"": 0}}
else:
    model_kwargs = {"torch_dtype": torch.bfloat16, "device_map": "auto"}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

pipe = hf_pipeline_fn(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    return_full_text=False,
)

llm    = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
parser = JsonOutputParser()
chain  = prompt | llm | parser

precision = "NF4 4-bit" if LOAD_IN_4BIT else "BF16"
logger.info("Model loaded: %s  precision=%s  GPU=%s", MODEL_ID, precision, torch.cuda.get_device_name(0))
print(f"LLM loaded   : {MODEL_ID}")
print(f"Precision    : {precision}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print("LCEL chain   : prompt | ChatHuggingFace | JsonOutputParser")


03:44:23  INFO      HTTP Request: HEAD https://huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:44:23  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Meta-Llama-3.1-8B-Instruct/d10aef7999a2b5ba950ab3974312feeedbfe0b77/config.json "HTTP/1.1 200 OK"
03:44:23  INFO      HTTP Request: HEAD https://huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:44:23  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Meta-Llama-3.1-8B-Instruct/d10aef7999a2b5ba950ab3974312feeedbfe0b77/tokenizer_config.json "HTTP/1.1 200 OK"
03:44:23  INFO      HTTP Request: GET https://huggingface.co/api/models/NousResearch/Meta-Llama-3.1-8B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
03:44:23  INFO      HTTP Request: GET https://huggingface.co

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

03:45:21  INFO      HTTP Request: HEAD https://huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
03:45:21  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Meta-Llama-3.1-8B-Instruct/d10aef7999a2b5ba950ab3974312feeedbfe0b77/generation_config.json "HTTP/1.1 200 OK"
03:45:21  INFO      HTTP Request: GET https://huggingface.co/api/resolve-cache/models/NousResearch/Meta-Llama-3.1-8B-Instruct/d10aef7999a2b5ba950ab3974312feeedbfe0b77/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


03:45:22  INFO      HTTP Request: HEAD https://huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:45:22  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Meta-Llama-3.1-8B-Instruct/d10aef7999a2b5ba950ab3974312feeedbfe0b77/config.json "HTTP/1.1 200 OK"
03:45:22  INFO      HTTP Request: HEAD https://huggingface.co/NousResearch/Meta-Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:45:22  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Meta-Llama-3.1-8B-Instruct/d10aef7999a2b5ba950ab3974312feeedbfe0b77/tokenizer_config.json "HTTP/1.1 200 OK"
03:45:22  INFO      HTTP Request: GET https://huggingface.co/api/models/NousResearch/Meta-Llama-3.1-8B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
03:45:22  INFO      HTTP Request: GET https://huggingface.co

## Cell group 4: AMLState and node functions

`AMLState` is the shared state passed through every node. Each node returns a partial state update.

In [6]:
class AMLState(TypedDict, total=False):
    query:            str
    retrieved:        list
    context:          str
    answer:           str
    citations:        list
    citation_errors:  list
    retries:          int
    guardrail_result: dict
    human_approved:   bool
    final_answer:     str


def retrieve_node(state: AMLState) -> dict:
    results = dense_sparse_retrieve(
        state['query'], vectorstore, bm25, clauses,
        k=TOP_K_RETRIEVAL, rrf_k=RRF_K,
    )
    return {
        'retrieved': results,
        'context':   format_context(results),
        'retries':   0,
    }


def generate_node(state: AMLState) -> dict:
    try:
        parsed    = chain.invoke({'query': state['query'], 'context': state['context']})
        answer    = parsed.get('answer', '')
        citations = parsed.get('citations', [])
    except Exception as exc:
        answer    = f'Generation error: {exc}'
        citations = []
    return {'answer': answer, 'citations': citations}


def validate_node(state: AMLState) -> dict:
    retrieved_ids = {r['clause_id'] for r in state.get('retrieved', [])}
    errors = [cid for cid in state.get('citations', []) if cid not in retrieved_ids]
    return {'citation_errors': errors, 'retries': state.get('retries', 0) + 1}


def guardrail_node(state: AMLState) -> dict:
    if GUARDRAILS_ON:
        from guardrails import AMLGuardrail
        guard = AMLGuardrail()
    else:
        from guardrails import NullGuardrail
        guard = NullGuardrail()
    result = guard.check_output(state)
    return {'guardrail_result': result}


def human_review_node(state: AMLState) -> dict:
    # Production: graph pauses here; compliance officer reviews before resuming
    # Notebook smoke test: auto-approve
    return {'human_approved': True, 'final_answer': state.get('answer', '')}


def route_validate(state: AMLState) -> str:
    if state.get('citation_errors') and state.get('retries', 0) < MAX_RETRIES:
        return 'retry'
    return 'pass'


def route_guardrail(state: AMLState) -> str:
    result = state.get('guardrail_result') or {}
    return 'abstain' if not result.get('passed', True) else 'pass'


print('AMLState and node functions defined.')

AMLState and node functions defined.


## Cell group 5: LangGraph StateGraph

Compiled with `interrupt_before=['human_review']` so the graph pauses for HITL approval in production.

In [7]:
builder = StateGraph(AMLState)

builder.add_node('retrieve',     retrieve_node)
builder.add_node('generate',     generate_node)
builder.add_node('validate',     validate_node)
builder.add_node('guardrail',    guardrail_node)
builder.add_node('human_review', human_review_node)

builder.set_entry_point('retrieve')
builder.add_edge('retrieve', 'generate')
builder.add_edge('generate', 'validate')
builder.add_conditional_edges(
    'validate',
    route_validate,
    {'retry': 'generate', 'pass': 'guardrail'},
)
builder.add_conditional_edges(
    'guardrail',
    route_guardrail,
    {'abstain': END, 'pass': 'human_review'},
)
builder.add_edge('human_review', END)

pipeline = builder.compile(interrupt_before=['human_review'])
print('LangGraph pipeline compiled.')
print('Flow: retrieve -> generate -> validate -> (retry|guardrail) -> (abstain|human_review) -> END')

LangGraph pipeline compiled.
Flow: retrieve -> generate -> validate -> (retry|guardrail) -> (abstain|human_review) -> END


## Cell group 6: Citation validation

Deterministic post-processing: extract `[clause_id]` tokens, check against retrieved set, flag obligation statements without any citation.

In [8]:
import re as _re

_CITATION_RE   = _re.compile(r'\[([a-z][a-z0-9_]*)\]')
_OBLIGATION_RE = _re.compile(r'\b(must|shall|required|obliged|obligation)\b', _re.IGNORECASE)


def extract_citations(text: str) -> list:
    return list(dict.fromkeys(_CITATION_RE.findall(text)))


def validate_citations(answer_text: str, retrieved: list) -> dict:
    retrieved_ids = {r['clause_id'] for r in retrieved}
    cited         = extract_citations(answer_text)
    hallucinated  = [cid for cid in cited if cid not in retrieved_ids]

    sentences = _re.split(r'(?<=[.!?])\s+', answer_text)
    uncited_obligations = [
        s for s in sentences
        if _OBLIGATION_RE.search(s) and not _CITATION_RE.search(s)
    ]

    return {
        'passed':              not hallucinated,
        'cited':               cited,
        'hallucinated':        hallucinated,
        'uncited_obligations': uncited_obligations,
    }


# Synthetic test
_ans = 'CDD must be applied when establishing a relationship [mlr_2017_reg_27]. EDD applies to PEPs [mlr_2017_reg_35].'
_ret = [{'clause_id': 'mlr_2017_reg_27'}, {'clause_id': 'mlr_2017_reg_35'}]
_v   = validate_citations(_ans, _ret)
assert _v['passed'], f'Expected pass: {_v}'
assert _v['hallucinated'] == []
assert _v['cited'] == ['mlr_2017_reg_27', 'mlr_2017_reg_35']
print('Citation validation: passed')
print('  cited:', _v['cited'])
print('  hallucinated:', _v['hallucinated'])
print('  uncited_obligations:', _v['uncited_obligations'])

Citation validation: passed
  cited: ['mlr_2017_reg_27', 'mlr_2017_reg_35']
  hallucinated: []
  uncited_obligations: []


## Cell group 7: Guardrails

Imports `NullGuardrail` and `AMLGuardrail` from `scripts/guardrails.py`. Runs verification with synthetic inputs.

In [9]:
from guardrails import NullGuardrail, AMLGuardrail, GuardrailResult

_null = NullGuardrail()
_aml  = AMLGuardrail()

_state_aml = {
    'query':     'customer due diligence requirements under MLR 2017',
    'retrieved': [{'clause_id': 'mlr_2017_reg_27', 'rrf_score': 0.05}],
    'answer':    'CDD is required [mlr_2017_reg_27].',
    'citations': ['mlr_2017_reg_27'],
}
_state_off = {
    'query':     'weather forecast for Edinburgh',
    'retrieved': [],
    'answer':    '',
    'citations': [],
}

assert _null.check_output(_state_off)['passed']
assert not _aml.check_input(_state_off).passed
assert _aml.check_input(_state_aml).passed
assert _aml.check_output(_state_aml)['passed']

print('NullGuardrail: always passes')
print('AMLGuardrail: rejects off-topic query, accepts AML query')
print('Guardrail verification passed.')

NullGuardrail: always passes
AMLGuardrail: rejects off-topic query, accepts AML query
Guardrail verification passed.


## Cell group 8: Smoke test

Retrieval stage only: no LLM required. Full pipeline (with LLM) shown as commented code at the bottom.

In [10]:
SMOKE_QUERIES = [
    'What are the customer due diligence requirements under MLR 2017?',
    'When must a firm submit a suspicious activity report?',
    'How are politically exposed persons defined and what enhanced measures apply?',
]

print('Smoke test: retrieval stage only (model loading not required)\n')

for query in SMOKE_QUERIES:
    retrieved = retrieve_node({'query': query})['retrieved']
    print(f'Query: {query!r}')
    for r in retrieved[:3]:
        rank  = r['rank']
        cid   = r['clause_id']
        score = r['rrf_score']
        print(f'  #{rank}  {cid:<44}  rrf={score:.5f}')
    print()

print('Retrieval stage verified.')
print()
print('To run the full pipeline (requires cell group 3 model loading):')
print('  result = pipeline.invoke({"query": SMOKE_QUERIES[0]})')
print('  # graph pauses at human_review, resume with:')
print('  # result = pipeline.invoke(None, config={"configurable": {"thread_id": "1"}})')

Smoke test: retrieval stage only (model loading not required)

03:45:27  INFO      dense_sparse_retrieve: query='What are the customer due diligence requirements under MLR 2'  top-1=jmlsg_2_18.93
Query: 'What are the customer due diligence requirements under MLR 2017?'
  #1  jmlsg_2_18.93                                 rrf=0.03202
  #2  mlr_2017_sch6_para7                           rrf=0.03126
  #3  jmlsg_1_5.1.1                                 rrf=0.01639

03:45:28  INFO      dense_sparse_retrieve: query='When must a firm submit a suspicious activity report?'  top-1=jmlsg_1_6.86
Query: 'When must a firm submit a suspicious activity report?'
  #1  jmlsg_1_6.86                                  rrf=0.03200
  #2  jmlsg_2_18.62                                 rrf=0.03151
  #3  fatf_40_R20                                   rrf=0.03012

03:45:28  INFO      dense_sparse_retrieve: query='How are politically exposed persons defined and what enhance'  top-1=fatf_40_R12
Query: 'How are political